# T2A — ViT / image classifier — isotropic sensor noise

**Lemma D2** · `nuisance="isotropic"` · [Task doc](../../docs/tasks/t02a-vit-isotropic.md) · FINAL: `paper_code/T2/Task2A/FINAL.md`

> Isotropic PMH on ViT-B/16: +4.29 pp mean ImageNet-C, TDI −58% at σ=0.10.

| § | What you do |
|---|-------------|
| 1–4 | Install → load demo → `check_applicability` |
| 5–6 | Estimate $\Sigma_{\text{task}}$ → PMH train → Step 5 on deploy holdout |
| 7–8 | Reproduce paper scripts → plug in your data |


**Demo note:** Mini RGB CNN (32×32), not full ImageNet ViT.


## 1 — Install


In [ ]:
!pip install -q matching-pmh torch


## 2 — Config & imports


In [ ]:
import os
import torch
from pmh.benchmark.presets import get_preset
from pmh.pytorch_eval import (
    pytorch_demo_loaders,
    pytorch_isotropic_demo_loaders,
    pytorch_multilayer_vision_demo_loaders,
    pytorch_sequence_demo_loaders,
)
from pmh import PMHConfig, PMHTrainer, evaluate_robust_fit, check_applicability, suggest_nuisance
from pmh.adoption import RECIPE_ONE_LINER, format_recipe_banner

QUICK = os.environ.get("PMH_QUICK", "").lower() in ("1", "true", "yes")
EPOCHS = 2 if QUICK else 6
SEED = 0
print(RECIPE_ONE_LINER)


## 3 — Load demo data


In [ ]:
preset = get_preset("t2a_vit_isotropic")
bundle = pytorch_isotropic_demo_loaders(n=N, batch_size=32, seed=SEED, eval_noise_sigma=0.10)
model = bundle.model
hook, head = bundle.encoder, bundle.head
train_loader, src_loader, val_loader = bundle.train_loader, bundle.source_batches, bundle.val_loader
tgt_loader = val_loader  # noisy deploy holdout (sigma=0.10)
print("demo", bundle.n_classes, "classes")


## 4 — Scope (applicability)


In [ ]:
from pmh import check_applicability, suggest_nuisance

print(suggest_nuisance(has_source_labels=True, has_target_domain=False))
app = check_applicability(stack="pytorch", has_target_domain=False)
print(app.summary())


## 5 — Estimate $\Sigma_{\text{task}}$ + PMH train


In [ ]:
import copy
from pmh import PMHTrainer, PMHConfig

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
m = copy.deepcopy(model).to(device)
trainer = PMHTrainer(
    m, hook=hook, head=head, nuisance="isotropic", noise_level=preset.estimate_kwargs["noise_level"], pmh_config=preset.pmh_config, device=device,
)
trainer.fit(train_loader, source_batches=src_loader, epochs=EPOCHS)
print("preflight", trainer.artifact_.preflight, "method", getattr(trainer.artifact_, "method", None))


## 6 — Step 5 (deploy holdout)


In [ ]:
from pmh import evaluate_robust_fit

report = evaluate_robust_fit(
    m, train_loader, val_loader,
    source_batches=src_loader, target_batches=tgt_loader,
    hook=m.enc, head=m.head, nuisance="isotropic", rank=16, 
    pmh_config=preset.pmh_config, epochs=max(2, EPOCHS - 2), include_falsification=False, seed=SEED,
    noise_level=preset.estimate_kwargs["noise_level"],
)
print(report.summary())
if hasattr(report, "baseline_metric"):
    print("deploy holdout — baseline:", report.baseline_metric, "pmh:", report.pmh_metric)


### Geometry probe (paper §5)


In [ ]:
from pmh.tdi import trajectory_tdi_encoder
tdi = trajectory_tdi_encoder(m, hook, src_loader, sigma=0.10, max_batches=8, device=device)
print("trajectory_tdi:", tdi.get("trajectory_tdi"))


## 7 — Paper reproduction


Frozen results: `paper_code/T2/Task2A/FINAL.md`

- **ImageNet ViT-B/16 + isotropic PMH (Type 2A):** `python paper_code/T2/Task2A/train.py`
- **TDI / Jacobian probes (label-free):** `python paper_code/T2/Task2A/recompute_task1a_tdi.py`
- **ImageNet-C transfer (15 corruptions, severity 3):** `python paper_code/T2/Task2A/eval_imagenet_c.py`


## 8 — Your pipeline


Swap demo loaders for your `train_loader`, `source_batches`, `target_batches`, and deploy holdout. Hook the backbone before your task head.
